# Owner and Transaction Scale Calculations
Created by Nicholas Polimeni

Updated by Melissa Juarez to include more data cleaning

This models after the fulton entity scale code but removes sales code and focuses only on ownership key creation.

In [5]:
import pandas as pd
import os
import re

pd.set_option('display.max_columns', 150)
pd.options.display.float_format = '{:,.3f}'.format

# set wd one folder back
os.chdir('/Users/melissajuarezc/Documents/GITHUB REPOS/parcel-data-processor/')

In [3]:
# Cleaned digest and sales data from clean_data.ipynb
FILES_PATH = 'output/gwinnett/'
digest_full = pd.read_csv(
    FILES_PATH + 'DIGEST_gwinnett_2011_2023.csv',
    dtype = {
    "tax_year": "Int64",
    "lrs_num": "Int64",
    "pin": "string",
    "public_nei_num": "Int64",
    "propertystreet": "string",
    "property_city": "string",
    "property_state": "string",
    "property_zip": "string",
    "ownername1": "string",
    "ownername2": "string",
    "owneraddress1": "string",
    "ownercity": "string",
    "ownerstate": "string",
    "ownerzip": "string",
    "legalac": "float64",
    "pcdesc": "string",
    "zonedesc": "string",
    "exempt1": "Int64",
    "exempt1d": "Int64",
    "assmnt1d": "string",
    "landval1": "float64",
    "dwlgval1": "float64",
    "othval1": "float64",
    "totval1": "float64",
    "taxland1": "float64",
    "taxdwlg1": "float64",
    "taxoth1": "float64",
    "taxtot1": "float64",
    "sale1d": "string",
    "sale2d": "string",
    "sale3d": "string",
    "sale1amt": "float64",
    "sale2amt": "float64",
    "sale3amt": "float64",
    "grantorname1": "string",
    "grantorname2": "string",
    "grantorname3": "string",
    "doc1ref": "string",
    "doc2ref": "string",
    "doc3ref": "string",
    "legal1": "string",
    "propclas": "float64",
    "distnum": "string",
    "distnum_desc": "string",
    "propclas_desc": "string", 
}
)

## Modify data to enable aggreggating on entity key

The goal for these scripts is to clean the ownership address columns so that we can identify properties where the owners are the same. Upon closer look of Gwinnett County's format for ownership data, there are some issues that were present in Cobb and Fulton that will not be an issue for Gwinnett:
-   PO BOX formats are consistent for all owners
-   Only 45 missing addresses
-   Addresses in standard formats
-   standard address suffix abbrevitions (dr, ct, xing, cv)


Goal: Create an Owner Address (labeled: "owner_addr") column that is the concatentation of owner address number, owner address string, owner unit number, and owner zip.

In [236]:
digest_full['mod_own_adrstr'] = digest_full['owneraddress1'].copy(deep=True)

## if there is a # character, put a space after it and then remove any double spaces with only one space.
digest_full['mod_own_adrstr'] = digest_full['mod_own_adrstr'].str.replace(r'#(?!\s)', '# ', regex=True)

## first, take out the apt/suite/ste and move to mod_unit_no 
unit_pattern = r'(?:\b(?:unit|apt|ste|suite|suites|bldg)\b|#)[\s\.\-]*(.*)$'
digest_full['mod_unitno'] = digest_full['mod_own_adrstr'].str.extract(unit_pattern, flags=re.IGNORECASE)

## remove unit pattern from original address
digest_full['mod_own_adrstr'] = digest_full['mod_own_adrstr'].str.replace(
    unit_pattern, '', flags=re.IGNORECASE, regex=True
).str.strip()

# clean mod_unitno: remove the keywords from mod_unitno to only retain the numbers/letters assoc w unit + other cleaning
digest_full['mod_unitno'] = (
    digest_full['mod_unitno']
    .str.replace(r'^(?:unit|apt|ste|suite|suites|bldg)[\s\.\-]*', '', flags=re.IGNORECASE, regex=True)
    .str.replace(r'[#\-]', '', regex=True)          # remove any remaining # or -
    .str.replace(r'\s+', '', regex=True)            # remove all spaces
    .fillna('')                                     # fill NA with empty string
)

In [237]:
digest_full[
    ["owneraddress1", "mod_own_adrstr", "mod_unitno"]
].sample(10)

,owneraddress1,mod_own_adrstr,mod_unitno
1256125,807 OLDE MILL LN,807 OLDE MILL LN,
535996,5109 WALDEN CROSSING DR,5109 WALDEN CROSSING DR,
907055,453 MACLAND DR,453 MACLAND DR,
2748656,4712 BRAMBLE ROSE LN,4712 BRAMBLE ROSE LN,
715493,700 NW 107TH AVE STE 200,700 NW 107TH AVE,200
112710,2171 COLONIAL OAK WAY,2171 COLONIAL OAK WAY,
3507690,2725 CABOT CT,2725 CABOT CT,
3034805,2462 BANCROFT WAY,2462 BANCROFT WAY,
99818,3874 LAURENS LN,3874 LAURENS LN,
2204026,1307 FRONTIER TRL,1307 FRONTIER TRL,


### Identify same owners in parcel data

Issues I am seeing:
-   some instances of crossing need to be changed to xing
-   APT/STE/SUITE/UNIT numbers need to be separated into new column (~19.4k observations)
-   Need to remove the address suffixes to be able to match with the other ownership key formats
-   clean ownerzip to only take the first 5 characters


**Why:** these values get us a highly accurate key for same owner. Owner address string does not contain postfixes like ST, AVE, etc. that might cause issues. Combined with owner number and owner zip, we can say with high confidence that the address is the same while avoiding many common differences amongst the same address (ST vs STREET, etc.). This method is prefered over names which has a higher chance of false positive, and large corporations may operate with differently named subsidaries. This method may also undercount, if a company uses multiple addresses, but this is somewhat unlikely and undercounting is simply an acceptable limitation. It is acceptable since large investors (who would use different addresses) will own so many properties with each subsidary that it will be binned in the correct bin regardless.

In [238]:
## Then remove directionals if they exist. The goal is to only have the street name in the ownership address key.
directional_pattern = r'\b(?:NW|NE|SW|SE|N|S|E|W)\b$'

digest_full['mod_own_adrsuf2'] = digest_full['mod_own_adrstr'].str.extract(
    f'({directional_pattern})', flags=re.IGNORECASE
).fillna('')

digest_full['mod_own_adrstr'] = digest_full['mod_own_adrstr'].str.replace(
    directional_pattern, '', flags=re.IGNORECASE, regex=True
).str.strip()

## replace crossing w xing only if it is last word in mod_own_adrstr
digest_full['mod_own_adrstr'] = digest_full['mod_own_adrstr'].str.replace(
    r'\bcrossing\b$', 'XING', flags=re.IGNORECASE, regex=True
)

# Next, remove address suffixes from mod_own_adrstr
suffixes = [
    "aly", "arc", "ave", "blvd", "bnd", "br", "brg", "btm", "byp", "cir", "clb", "clfs",
    "cmns", "cor", "cors", "crk", "cres", "crse", "cswy", "ct", "ctr", "ctrs", "cts",
    "cv", "cvs", "cyn", "dr", "drs", "est", "ests", "expy", "ext", "exts", "fld", "flds",
    "flt", "flts", "frd", "frds", "frg", "frgs", "frk", "frks", "ft", "fwy", "gdn", "gdns",
    "glf", "grn", "grns", "grv", "gtwy", "hbr", "hl", "hls", "holw", "hwy", "inlt", "is",
    "iss", "jct", "jcts", "key", "knl", "knls", "lk", "lks", "ln", "lndg", "loop", "mall",
    "mnr", "mnrs", "mdw", "mdws", "ml", "mls", "msn", "mt", "mts", "nck", "opas", "orch",
    "oval", "park", "pass", "path", "pike", "pkwy", "pl", "pln", "plns", "plz", "pt",
    "pts", "radl", "ramp", "rd", "rdg", "rdgs", "rds", "riv", "rnch", "row", "rpd", "rpds",
    "rst", "rte", "rue", "run", "shl", "shls", "shr", "shrs", "skwy", "smt", "spg", "spgs",
    "spur", "sq", "sqs", "st", "sta", "stra", "strm", "ter", "terr", "trce", "trfy",
    "trl", "tunnl", "un", "uns", "vly", "vlg", "vlgs", "vlly", "vis", "walk", "wall",
    "way", "well", "wls", "xing"
]

# create pattern to match any suffix at the END of string
suffix_pattern = r'\b(?:' + '|'.join(suffixes) + r')\b$'

digest_full['mod_own_adrsuf'] = digest_full['mod_own_adrstr'].str.extract(
    f'({suffix_pattern})', flags=re.IGNORECASE
).fillna('')

# remove only if it's the last word
digest_full['mod_own_adrstr'] = digest_full['mod_own_adrstr'].str.replace(
    suffix_pattern, '', flags=re.IGNORECASE, regex=True
).str.strip()



In [239]:
digest_full[
    ["owneraddress1", "mod_own_adrstr","mod_own_adrsuf", "mod_own_adrsuf2", "mod_unitno"]
].sample(10)

,owneraddress1,mod_own_adrstr,mod_own_adrsuf,mod_own_adrsuf2,mod_unitno
1503486,664 GARNER RD SW,664 GARNER,RD,SW,
868709,1770 MISSION PARK CT,1770 MISSION PARK,CT,,
3385370,4912 SUMMER OAK DR,4912 SUMMER OAK,DR,,
2898661,75 LANGLEY DR,75 LANGLEY,DR,,
2410052,3370 COMMONS GATE BND,3370 COMMONS GATE,BND,,
2333780,4595 ASHINGTON DR,4595 ASHINGTON,DR,,
99817,1101 WOODED ACRES,1101 WOODED ACRES,,,
2365362,28 HARMONY GROVE RD,28 HARMONY GROVE,RD,,
1744160,1132 TIMBER CREEK DR,1132 TIMBER CREEK,DR,,
943986,3755 COLONIAL TRL SW,3755 COLONIAL,TRL,SW,


In [223]:
# Print total number of PO BOXES without a number in their address string
re_po_box_no_number = r"^(?!.*\d)[P]+.* BOX.*"
len(digest_full[digest_full["mod_own_adrstr"].str.contains(
    re_po_box_no_number, regex=True, na=False
)][["owneraddress1", "mod_own_adrstr"]])

35

In [240]:
## clean owner zip to keep first 5 chars
digest_full['mod_ownerzip'] = digest_full['ownerzip'].astype(str).str[:5]

# regex to clean by replacing dots, commas, and multiple spaces
# make all strings uppercase (they should be already)

re_dots_commas = r"[.,]+"
re_multiple_spaces = r"\s{2,}"

digest_full["owner_addr"] = (
    digest_full["mod_own_adrstr"] + " " +
    digest_full["mod_unitno"].fillna('') + " " +
    digest_full["mod_ownerzip"].astype("string").fillna('')
).str.replace(
    re_dots_commas,
    "",
    regex=True
).str.replace(
    re_multiple_spaces,
    " ",
    regex=True
).str.strip().str.upper()

In [241]:
digest_full[
    ["owneraddress1", "ownerzip", "mod_own_adrstr", "mod_unitno", "mod_ownerzip", "owner_addr"]
].sample(10)

,owneraddress1,ownerzip,mod_own_adrstr,mod_unitno,mod_ownerzip,owner_addr
3285140,3046 MEADOW LARK DR,30096-3948,3046 MEADOW LARK,,30096,3046 MEADOW LARK 30096
1929084,502 WINTER HAVEN LN,30518-7701,502 WINTER HAVEN,,30518,502 WINTER HAVEN 30518
798846,11 CHERRYSTONE CT,30024-2380,11 CHERRYSTONE,,30024,11 CHERRYSTONE 30024
2238747,1555 QUAIL POINT RUN,30548-1639,1555 QUAIL POINT,,30548,1555 QUAIL POINT 30548
3659005,3504 WHITE SANDS WAY,30024-7056,3504 WHITE SANDS,,30024,3504 WHITE SANDS 30024
3462790,961 LISA KAY DR,30046-8353,961 LISA KAY,,30046,961 LISA KAY 30046
308903,3201 HAMPTON RIDGE WAY,30078-3884,3201 HAMPTON RIDGE,,30078,3201 HAMPTON RIDGE 30078
2509026,460 EMERALD LAKE PATH,30518-5622,460 EMERALD LAKE,,30518,460 EMERALD LAKE 30518
3618191,1330 SEVER WOODS DR,30043-6240,1330 SEVER WOODS,,30043,1330 SEVER WOODS 30043
3387607,4960 PRICE DR,30024-4186,4960 PRICE,,30024,4960 PRICE 30024


## Identify corporate owners, create corp owner flags for each record
- grantee, grantor in sales
- own1 in digest

In [242]:
# Any with risk of false positive like "CO" need to have a space prepended or postpended
corp_keywords = [
    'LLC', ' INC', 'LLP', 'L.L.C', 'L.L.P', 'I.N.C', 'L L C',
    'L L P', ' L P', ' LP', 'LTD', ' CORP', 'CORPORATION',
    'COMPANY', ' CO ', 'LIMITED', 'PARTNERSHIP', 'PARTNERSHIPS',
    'ASSOCIATION', 'ASSOC', 'INCORPORATED', 'INCORP',
    'L.T.D', 'LTD', "HOME", "SOLUTIONS"
]

# Make a list of all corp owners -- added own2 as well.
corps = digest_full[
    digest_full["ownername1"].apply(lambda x: any([key in str(x) for key in corp_keywords]))
]['ownername1'].unique().tolist() + digest_full[
    digest_full["ownername2"].apply(lambda x: any([key in str(x) for key in corp_keywords]))
]['ownername2'].unique().tolist()

with open("./output/gwinnett/corp_names.txt", "w") as f:
    f.write("\n".join(corps))

In [243]:
digest_full["own_corp_flag"] = (
    digest_full["ownername1"].isin(corps) | digest_full["ownername2"].isin(corps)
).astype(int)

digest_full[['ownername1', 'ownername2', 'own_corp_flag']].sample(10)

,ownername1,ownername2,own_corp_flag
2402736,LIVINGSTON DAVID,CARLIN DAVID CHARLES,0
1907944,SHAH RAJAN R,SHAH MANALI R,0
2101440,SHIN PAUL,<NA>,0
1530576,KAMUTY LIA MARK,<NA>,0
2683364,POPE TRACY,<NA>,0
2916564,PROGRESS RESIDENTIAL BORROWER 5 LLC,<NA>,1
1466929,2014-1 IH BORROWER LP,<NA>,1
1639887,JACKSON KELLI DEE,<NA>,0
1308356,SHAFFER SUE ANN,SHAFFER JAMES BRYAN,0
795023,THR GEORGIA LP,<NA>,1


## Create a rental property flag

In [ ]:
# when owner address is not the same as property address -- issues here with Crossing vs Xing in raw data @mel 07.02.25
digest_full["rental_flag"] = 0

digest_full["mod_owneraddress1"] = digest_full["mod_own_adrstr"] + " " + digest_full["mod_own_adrsuf"] + " " + digest_full["mod_own_adrsuf2"]
digest_full["mod_owneraddress1"] = digest_full["mod_owneraddress1"].str.strip()

## second option where suf2 is not included.
digest_full["mod_owneraddress1_B"] = digest_full["mod_own_adrstr"] + " " + digest_full["mod_own_adrsuf"]
digest_full["mod_owneraddress1_B"] = digest_full["mod_owneraddress1_B"].str.strip()

digest_full.loc[
    ((digest_full["propertystreet"] != digest_full["mod_owneraddress1"]) & (digest_full["propertystreet"] != digest_full["mod_owneraddress1_B"])),
    "rental_flag"
] = 1

In [259]:
digest_full[['propertystreet', 'mod_owneraddress1', 'mod_owneraddress1_B', 'mod_own_adrstr', 'mod_own_adrsuf', 'mod_own_adrsuf2', 'rental_flag']].sample(20)

,propertystreet,mod_owneraddress1,mod_owneraddress1_B,mod_own_adrstr,mod_own_adrsuf,mod_own_adrsuf2,rental_flag
660507,HWY 78 W,2 CAPITOL SQ SW,2 CAPITOL SQ,2 CAPITOL,SQ,SW,1
407405,265 TAMBEC TRCE,265 TAMBEC TRCE NW,265 TAMBEC TRCE,265 TAMBEC,TRCE,NW,0
1257121,5588 ESTATES CT,PO BOX 670672,PO BOX 670672,PO BOX 670672,,,1
2640106,3209 WOOD SPRINGS CT,3209 WOOD SPRINGS CT SW,3209 WOOD SPRINGS CT,3209 WOOD SPRINGS,CT,SW,0
868596,3908 ROSEBUD RD,3908 ROSEBUD RD,3908 ROSEBUD RD,3908 ROSEBUD,RD,,0
1971407,760 FOREST OAK DR,760 FOREST OAK DR,760 FOREST OAK DR,760 FOREST OAK,DR,,0
1634723,2875 PEACHTREE INDUSTRIAL BLVD,1110 SATELLITE BLVD NW,1110 SATELLITE BLVD,1110 SATELLITE,BLVD,NW,1
353556,1162 SIMONTON GLEN WAY,1162 SIMONTON GLEN WAY,1162 SIMONTON GLEN WAY,1162 SIMONTON GLEN,WAY,,0
2278765,1531 EDGELEY WAY,1531 EDGELEY WAY,1531 EDGELEY WAY,1531 EDGELEY,WAY,,0
437353,2648 DAVENHAM LN,68 SOUTHLAKE DR,68 SOUTHLAKE DR,68 SOUTHLAKE,DR,,1


## Create ownership scale table

In [265]:
owned_gwinnett_yr = pd.DataFrame(
    digest_full.groupby(["tax_year", "owner_addr"])["pin"].count()
).rename(columns={"pin": "count_owned_gwinnett_yr"}).reset_index()
owned_gwinnett_yr

assoc_owner_names = pd.DataFrame(
    digest_full.groupby(["owner_addr"]).agg({"ownername1": list})
).rename(columns={"ownername1": "assoc_owner_names"}).reset_index()

owner_scale = owned_gwinnett_yr.merge(
    assoc_owner_names,
    on=["owner_addr"],
    how="left"
)

owner_scale.sort_values(by="count_owned_gwinnett_yr", ascending=False).head(5)

,tax_year,owner_addr,count_owned_gwinnett_yr,assoc_owner_names
2284686,2021,1717 MAIN 2000 75201,3011,"[2015-2 IH2 BORROWER LP, IH6 PROPERTY GEORGIA ..."
2050628,2020,1717 MAIN 2000 75201,2995,"[2015-2 IH2 BORROWER LP, IH6 PROPERTY GEORGIA ..."
2759874,2023,1717 MAIN 2000 75201,2876,"[2015-2 IH2 BORROWER LP, IH6 PROPERTY GEORGIA ..."
2521327,2022,1717 MAIN 2000 75201,2820,"[2015-2 IH2 BORROWER LP, IH6 PROPERTY GEORGIA ..."
1819515,2019,1717 MAIN 2000 75201,2722,"[2015-2 IH2 BORROWER LP, IH6 PROPERTY GEORGIA ..."


In [266]:
owner_scale[
    owner_scale["tax_year"] == 2023
].sort_values(by="count_owned_gwinnett_yr", ascending=False).head(15)

,tax_year,owner_addr,count_owned_gwinnett_yr,assoc_owner_names
2759874,2023,1717 MAIN 2000 75201,2876,"[2015-2 IH2 BORROWER LP, IH6 PROPERTY GEORGIA ..."
2793962,2023,23975 PARK SORRENTO 300 91302,1589,"[AMERICAN HOMES 4 RENT PROPERTIES EIGHT LLC, A..."
2956529,2023,PO BOX 4090 85261,1391,"[PROGRESS RESIDENTIAL 2015-1 BORROWER, PROGRES..."
2766350,2023,1850 PARKWAY 900 30067,920,"[CERBERUS SFR HOLDINGS LP, CERBERUS SFR HOLDIN..."
2730811,2023,120 S RIVERSIDE 2000 60606,826,"[HPA US1 LLC, HP GEORGIA I LLC, HPA US1 LLC, H..."
2946256,2023,8665 E HARTFORD 200 85255,724,"[CSH 2016-1 BORROWER, LLC, CAH 2015-1 BORROWER..."
2938776,2023,75 LANGLEY 30046,660,"[GWINNETT COUNTY BOARD OF COMMISSIONE, GWINNET..."
2749236,2023,1508 BROOKHOLLOW 92705,494,"[TAH 2016-1 BORROWER LLC, TAH 2016-1 BORROWER ..."
2901136,2023,5001 PLAZA ON THE 200 78746,473,"[HFS I ASSETS COMPANY LLC, HFS I ASSETS COMPA..."
2947136,2023,8800 E RAINTREE 85260,458,"[MERITAGE HOMES OF GEORGIA INC, MERITAGE HOMES..."


### Identify major institutional investors

In [267]:
import re

owner_keywords = {
    "Amherst": ["AMHERST", "ARVM"],
    "Cerberus": ["CERBERUS", "FKH", "RM1 ", "RMI "],
    "Progress": ["PROGRESS", "FYR"],
    "Invitation": ["INVITATION", "IH "],
    "Colony": ["COLONY", "STARWOOD", "CSH", "CAH "],
    "Sylvan": ["SYLVAN", "RNTR"],
    "Tricon": ["TRICON", "TAH"]
}

for owner in owner_keywords:
    query_str = "|".join(owner_keywords[owner])

    filtered_rows = owner_scale[
        (owner_scale["tax_year"] == 2020) &
        owner_scale["assoc_owner_names"].apply(lambda x: any(
            ((re.search(query_str, name)) for name in x)
        ))
    ]
    
    filtered_rows = filtered_rows[
        filtered_rows["count_owned_gwinnett_yr"] > 49
    ]
    
    total = filtered_rows["count_owned_gwinnett_yr"].sum()
    print(f"{owner} owned {total} properties in 2020")
    display(filtered_rows)
    
    addresses = filtered_rows["owner_addr"].unique().tolist()
    print(addresses)
    
    names = [set(x) for x in filtered_rows[
        filtered_rows["count_owned_gwinnett_yr"] > 49
    ]["assoc_owner_names"].to_list()]

    names = set().union(*names)
    names = ", ".join(names)

    with open(f"./output/gwinnett/ownership_scale/{owner}_names.txt", "w") as f:
        f.write("KEYWORDS: " + query_str + "\n")
        f.write("ADDRESSES: " + ", ".join(addresses) + "\n\n")
        f.write("NAMES\n--------------------\n")
        f.write(names)

TypeError: expected string or bytes-like object, got 'NAType'

## Save owner scale

In [ ]:
OUTPUT_PATH = 'output/gwinnett/'

owner_scale.to_csv(OUTPUT_PATH + 'ownership_scale/owner_scale.csv', index=False)

OSError: Cannot save file into a non-existent directory: 'output/cobb/ownership_scale'

## Save

In [268]:
OUTPUT_PATH = 'output/gwinnett/'

digest_full.to_csv(OUTPUT_PATH + 'gwinnett_digest_full_final.csv', index=False)